## HW5 — оценка моделей на holdout и cross-validation

Проверим, как линейная регрессия переносится на отложенные данные. Сравним исходные числовые признаки Taxi и ранее подготовленные признаки. Основа — `Cross-Valodation/HW5_NEW.ipynb`; порядок упражнений сохранён. Полный CSV здесь не пересчитывался, исторические outputs не выдаются за актуальные результаты.

### Условия сравнения

Для обеих моделей используйте один и тот же test размером 20% и одинаковые разбиения оставшихся данных на 20 фолдов. Измеряйте MSLE: средний квадрат разностей логарифмов `1 + trip_duration`, без квадратного корня. Для такой оценки обучайте линейную регрессию на `log1p(trip_duration)` и считайте MSE в логарифмической шкале.

### Модель 1: исходные числовые признаки

Выберите перечисленные числовые колонки и подготовьте таргет для обучения.

In [1]:
import pandas as pd
from sklearn.svm import LinearSVC

In [ ]:
initial_data = pd.read_csv('initial_data.csv', index_col='id')

initial_cols = ['vendor_id', 'passenger_count', 'pickup_longitude',
                'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
                'trip_duration']

initial_data = initial_data[initial_cols]

### Логарифмирование таргета

In [ ]:
### Замерять будем MSLE. Можно показать, что для оптимизации MSLE,
### Достаточно логарифмировать таргетную переменную,
### а потом оптимизировать привычные MSE
import numpy as np

initial_data = initial_data.assign(log_trip_duration=np.log1p(initial_data['trip_duration']))
initial_data = initial_data.drop('trip_duration', axis=1)

### Почему здесь используется MSE

Обозначим `z = log(1 + y)`, а прогноз в этой шкале — `g(X)`. Тогда ошибка равна среднему `(z - g(X))²`. После обратного преобразования `a(X) = exp(g(X)) - 1` это выражение совпадает с формулой MSLE. Повторно логарифмировать уже преобразованный таргет при подсчёте метрики не нужно.

In [ ]:
### Выделим test

from sklearn.model_selection import train_test_split

X = initial_data.drop('log_trip_duration', axis=1)
y = initial_data['log_trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

In [ ]:
### Применим K-Fold на оставшуюяся валидационную часть X_train, y_train

from sklearn.model_selection import KFold

splitter = KFold(n_splits=20, shuffle=True, random_state=33)

In [ ]:
### Замерьте качество на кросс-валидации!

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

# Инициализируем регрессор
model_cv = LinearRegression()

# MSE уже логарифмированного таргета соответствует MSLE исходной длительности.
cv_scores = -cross_val_score(
    model_cv, X_train, y_train, cv=splitter, scoring="neg_mean_squared_error"
)

# Выводим среднее значение для задания 4
print(f"Среднее качество на CV (округлите до тысячных): {cv_scores.mean():.3f}")
### Your code is here

In [ ]:
### Теперь построим модель на всей тренировочной выборке
### и замерим качество на тесте!
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

# Строим регрессор на всей тренировочной выборке
model = LinearRegression()
model.fit(X_train, y_train)

# Предсказываем значения для тестовой выборки
y_pred = model.predict(X_test)

# Считаем финальный MSLE на тесте
test_score = mean_squared_error(y_test, y_pred)

print(f"Качество на тесте (округлите до тысячных): {test_score:.3f}")

### Your code is here

In [ ]:
### Укажите в ответе на задание 4 среднее качество моделей на валидационных выборках
### и качество модели, обученной на полной тренировочной выборке, на тестовой выборке.
### В качестве разделителя используйте точку, ответ округлите до тысячных.

### Модель 2: подготовленные признаки

Повторите оценку для обработанной таблицы. Сохраните состав и порядок объектов в train/test, чтобы тот же KFold выделял те же объекты. Незавершённые ячейки оценки ниже оставлены для самостоятельного решения.

In [ ]:
processed_data = pd.read_csv('processed_data.csv', index_col='id')

In [ ]:
processed_data.head()

In [ ]:
### Замерять будем MSLE. Можно показать, что для оптимизации MSLE,
### Достаточно логарифмировать таргетную переменную,
### а потом оптимизировать привычные MSE
import numpy as np

if 'trip_duration' in processed_data.columns:
    processed_data = processed_data.assign(log_trip_duration=np.log1p(processed_data['trip_duration']))
    processed_data = processed_data.drop('trip_duration', axis=1)

In [ ]:
X_2 = processed_data.drop('log_trip_duration', axis=1)
y_2 = processed_data['log_trip_duration']

In [ ]:
### Важно! Когда сравниваем модели по их качеству
### на валидации и на тесте, не шаффлим данные заново!

test_indexes = X_test.index
train_indexes = X_train.index

X_train_2 = X_2.loc[train_indexes]
y_train_2 = y_2.loc[train_indexes]

X_test_2 = X_2.loc[test_indexes]
y_test_2 = y_2.loc[test_indexes]

In [ ]:
### Замерьте качество на кросс-валидации!
### Your code is here

In [ ]:
### Теперь построим модель и замерим качество на тесте!
### Your code is here

In [ ]:
### Укажите в ответе на задание 5 среднее качество моделей на валидационных выборках
### и качество модели, обученной на полной тренировочной выборке, на тестовой выборке.
### В качестве разделителя используйте точку, ответ округлите до тысячных.

### Сравнение моделей

Какую из двух моделей выбрали бы вы? Помогла ли ранее выполненная подготовка признаков? Обоснуйте ответ по оценкам CV и test.